# SNCB/NMBS - Analyse Ponctualite Temps Reel

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import requests, zipfile, io, warnings
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
import folium
from folium.plugins import MarkerCluster, HeatMap
from IPython.display import display, HTML
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/clean", exist_ok=True)
print("Imports OK")

## 2. GTFS Statique

In [ ]:
GTFS_URL = "https://gtfs.irail.be/nmbs/gtfs/latest.zip"
try:
    resp = requests.get(GTFS_URL, timeout=120)
    resp.raise_for_status()
    zip_file = zipfile.ZipFile(io.BytesIO(resp.content))
    dataframes = {}
    for n in zip_file.namelist():
        if n.endswith(".txt"): dataframes[n.replace(".txt","")] = pd.read_csv(zip_file.open(n))
    for k,v in dataframes.items(): print(f"   {k}: {len(v):,}")
except Exception as e:
    print(f"Erreur: {e}")
    dataframes = {}

## 3. API iRail Temps Reel

In [ ]:
from src.realtime_api import RealtimeAPI
api = RealtimeAPI()
health = api.check_api_health()
print("iRail:", "OK" if health["irail"] else "DOWN")
print("GTFS-RT:", "OK" if health["gtfs_rt"] else "DOWN")

In [ ]:
stations = api.get_stations()
print(f"Gares: {len(stations)}")

In [ ]:
MAJOR_STATIONS = ["Brussels-Central","Brussels-South","Brussels-North","Gent-Sint-Pieters","Antwerpen-Centraal","Liege-Guillemins","Charleroi-Central","Bruges","Leuven","Namur","Mons","Mechelen","Hasselt","Kortrijk","Ostend"]
all_lb = api.get_liveboards_parallel(MAJOR_STATIONS)
print(f"Departs: {len(all_lb)}")

In [ ]:
disturbances = api.get_disturbances()
print(f"Perturbations: {len(disturbances)}")

In [ ]:
vp = api.vehicle_positions_to_df()
print(f"GTFS-RT positions: {len(vp)}")

## 4. KPIs

In [ ]:
from src.kpi_calculator import KPICalculator
if not all_lb.empty:
    td = all_lb.rename(columns={"vehicle_shortname":"route_id","delay_min":"arrival_delay"}).copy()
    td["trip_id"] = td["vehicle"]
    print(KPICalculator(td).generate_summary())

## 5. Carte (coords reelles)

In [ ]:
from src.map_generator import MapGenerator
map_gen = MapGenerator()
map_gen.create_base_map()
if not stations.empty and not all_lb.empty:
    sc = stations.set_index("name")[["locationX","locationY"]]
    tp = []
    for _,r in all_lb.iterrows():
        if r["station"] in sc.index:
            tp.append({"trip_id":r.get("vehicle",""),"route_id":r.get("vehicle_shortname","IC"),"latitude":float(sc.loc[r["station"],"locationY"]),"longitude":float(sc.loc[r["station"],"locationX"]),"departure_delay":r["delay_min"],"arrival_delay":r["delay_min"]})
    tp = pd.DataFrame(tp)
    if not tp.empty:
        map_gen.add_train_positions(tp)
        map_gen.add_legend()
        display(HTML(map_gen.get_map()._repr_html_()))